# FFA-MPDV Inspired Deepfake Detection (Kaggle)

This notebook implements a practical PyTorch pipeline inspired by the paper's FFA-MPDV idea: Meso4 + Capsule + FPN + Spatial Attention for binary deepfake detection.

In [ ]:
# Top-level runtime controls for Kaggle/local reruns
SMOKE_TEST = False

# Set to your actual dataset root for deterministic reruns.
# Keep blank to auto-pick from DATASET_ROOT_CANDIDATES when available.
DATASET_ROOT = ''

# Common Kaggle roots checked when DATASET_ROOT is empty.
DATASET_ROOT_CANDIDATES = [
    '/kaggle/input/deepfake-and-real-images/Dataset',
    '/kaggle/input/deepfake-and-real-images',
    '/kaggle/input/dfdc-subset',
    '/kaggle/input/dfdc',
    '/kaggle/input/faceforensics++',
]

# If split folders (train/val/test) are discovered, keep them instead of re-splitting.
PRESERVE_DISCOVERED_SPLITS = True

# Training profile options:
# - 'reproduce_paper_baseline': Meso4+FPN+Capsule, Adam, no scheduler, MSE
# - 'stable_modern': Meso4+FPN+Capsule (+ optional SegFormer), AdamW, cosine, BCE
TRAINING_PROFILE = 'reproduce_paper_baseline'

# Hard lock: force paper-faithful behavior even if other flags are changed later.
FORCE_PAPER_FAITHFUL = True

# SegFormer must stay OFF for paper-faithful reproduction.
ENABLE_SEGFORMER = False

# Paper recipe controls.
# Paper text has conflicting LR mentions (0.001 and 0.0001).
PAPER_RECIPE = True
PAPER_LR = 1e-4

# Training resilience controls for long full-dataset runs.
RESUME_TRAINING = True
MAX_TRAIN_HOURS = 0
CHECKPOINT_EVERY_N_BATCHES = 1200

# Resource pressure controls.
PIN_MEMORY = True
CPU_THREADS = 4
VAL_EVERY_N_EPOCHS = 1
SHOW_BATCH_PROGRESS = True

# Optional training profile note for tracking.
RUN_TAG = 'ffa_mpdv_paper_repro'

## Section 1: Environment and Imports
### Block 1.1 - Load required libraries and utilities

In [ ]:
import os
import gc
import random
import math
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

import matplotlib
from IPython import get_ipython

# Force notebook-friendly plotting. Use non-GUI backend when not in notebook.
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic('matplotlib', 'inline')
else:
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

try:
    import timm
    TIMM_AVAILABLE = True
except Exception:
    TIMM_AVAILABLE = False

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('timm available:', TIMM_AVAILABLE)
print('Matplotlib backend:', matplotlib.get_backend())


## Section 2: Configuration and Reproducibility
### Block 2.1 - Define config, seeds, and runtime device

In [ ]:
@dataclass
class CFG:
    seed: int = 42
    img_size: int = 256
    batch_size: int = 16
    num_workers: int = 2
    epochs: int = 50
    lr: float = 1e-4
    weight_decay: float = 1e-4
    use_mse_loss: bool = False
    val_size: float = 0.2
    threshold: float = 0.5
    early_stopping_patience: int = 8
    min_epochs_before_stop: int = 10

    # Finetuning controls
    optimizer_name: str = 'adamw'  # adam | adamw | sgd
    scheduler_name: str = 'cosine'  # none | cosine | plateau
    min_lr: float = 1e-6
    grad_clip_norm: float = 1.0
    momentum: float = 0.9

    # Run controls
    smoke_test: bool = bool(globals().get('SMOKE_TEST', False))
    smoke_max_train: int = 256
    smoke_max_val: int = 128
    smoke_min_per_class: int = 32
    smoke_force_balanced: bool = True

    # Resilience controls
    resume_training: bool = bool(globals().get('RESUME_TRAINING', True))
    max_train_hours: float = float(globals().get('MAX_TRAIN_HOURS', 0))
    checkpoint_every_n_batches: int = int(globals().get('CHECKPOINT_EVERY_N_BATCHES', 1200))

    # Resource controls
    pin_memory: bool = bool(globals().get('PIN_MEMORY', True))
    cpu_threads: int = int(globals().get('CPU_THREADS', 4))
    val_every_n_epochs: int = int(globals().get('VAL_EVERY_N_EPOCHS', 1))
    show_batch_progress: bool = bool(globals().get('SHOW_BATCH_PROGRESS', False))

    # Experiment controls
    training_profile: str = str(globals().get('TRAINING_PROFILE', 'reproduce_paper_baseline')).strip().lower()
    preserve_discovered_splits: bool = bool(globals().get('PRESERVE_DISCOVERED_SPLITS', True))

    # SegFormer integration
    use_segformer: bool = bool(globals().get('ENABLE_SEGFORMER', False))
    segformer_variant: str = 'mit_b0'
    segformer_pretrained: bool = True
    segformer_trainable: bool = False

    # Optional manual dataset root
    manual_data_root: str = str(globals().get('DATASET_ROOT', '')).strip()

    # Paper recipe controls
    paper_recipe: bool = bool(globals().get('PAPER_RECIPE', True))
    paper_lr: float = float(globals().get('PAPER_LR', 1e-4))

cfg = CFG()

profile = cfg.training_profile
if profile not in {'reproduce_paper_baseline', 'stable_modern'}:
    warnings.warn(f"Unknown TRAINING_PROFILE='{cfg.training_profile}'. Falling back to 'stable_modern'.")
    profile = 'stable_modern'

if profile == 'reproduce_paper_baseline':
    cfg.optimizer_name = 'adam'
    cfg.scheduler_name = 'none'
    cfg.use_mse_loss = bool(cfg.paper_recipe)
    cfg.lr = float(cfg.paper_lr)
    cfg.batch_size = 16
    cfg.threshold = 0.5
    cfg.use_segformer = False
else:
    cfg.optimizer_name = 'adamw'
    cfg.scheduler_name = 'cosine'
    cfg.use_mse_loss = False
    cfg.lr = min(cfg.lr, 1e-4)

# Optional hard lock to keep the run paper-faithful regardless of external flag changes.
if bool(globals().get('FORCE_PAPER_FAITHFUL', False)):
    profile = 'reproduce_paper_baseline'
    cfg.training_profile = 'reproduce_paper_baseline'
    cfg.optimizer_name = 'adam'
    cfg.scheduler_name = 'none'
    cfg.use_mse_loss = True
    cfg.lr = float(cfg.paper_lr)
    cfg.batch_size = 16
    cfg.threshold = 0.5
    cfg.use_segformer = False

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
is_kaggle = Path('/kaggle/input').exists() or ('KAGGLE_URL_BASE' in os.environ)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# If no explicit DATASET_ROOT is set, choose the first existing candidate (Kaggle-first).
if not cfg.manual_data_root:
    for cand in globals().get('DATASET_ROOT_CANDIDATES', []):
        if Path(cand).exists():
            cfg.manual_data_root = str(Path(cand))
            break

if cfg.smoke_test:
    cfg.epochs = min(cfg.epochs, 3)
    cfg.batch_size = min(cfg.batch_size, 8)
    cfg.early_stopping_patience = min(cfg.early_stopping_patience, 2)
    cfg.min_epochs_before_stop = 1
    cfg.use_mse_loss = False
    cfg.optimizer_name = 'adamw'
    cfg.scheduler_name = 'none'
    cfg.lr = min(cfg.lr, 3e-4)

# Worker defaults by environment: Kaggle can usually handle a couple workers,
# Windows local runs stay at single-process for stability.
if is_kaggle:
    cfg.num_workers = max(cfg.num_workers, 2)
elif os.name == 'nt':
    cfg.num_workers = 0

cfg.val_every_n_epochs = max(1, int(cfg.val_every_n_epochs))
if cfg.cpu_threads > 0:
    try:
        torch.set_num_threads(int(cfg.cpu_threads))
    except Exception as e:
        warnings.warn(f'Unable to set torch CPU threads: {e}')

print('Device:', device)
print('Kaggle environment:', is_kaggle)
print('Smoke test mode:', cfg.smoke_test)
print('Training profile:', profile)
print('Paper-faithful lock:', bool(globals().get('FORCE_PAPER_FAITHFUL', False)))
print('Preserve discovered splits:', cfg.preserve_discovered_splits)
print('Resolved DATASET_ROOT:', cfg.manual_data_root if cfg.manual_data_root else '(auto-discovery)')
print('SegFormer enabled:', cfg.use_segformer)
print('Optimizer:', cfg.optimizer_name, '| Scheduler:', cfg.scheduler_name)
print('Paper recipe:', cfg.paper_recipe, '| LR:', cfg.lr, '| MSE loss:', cfg.use_mse_loss)
print('Resume training:', cfg.resume_training, '| Max train hours:', cfg.max_train_hours)
print('Batch snapshot interval:', cfg.checkpoint_every_n_batches)
print('Pin memory:', cfg.pin_memory, '| CPU threads:', cfg.cpu_threads, '| Val every N epochs:', cfg.val_every_n_epochs)
print('Batch progress bars:', cfg.show_batch_progress)
print('Num workers:', cfg.num_workers)
print('Epochs:', cfg.epochs, '| Early stopping patience:', cfg.early_stopping_patience, '| Min epochs:', cfg.min_epochs_before_stop)
print('Resume note: to continue after Kaggle restarts, keep checkpoints in /kaggle/working/checkpoints and Save Version periodically.')

## Kaggle Runbook (Execute Top to Bottom)

Paper-faithful mode is now locked by default in this notebook.

Use this exact order on Kaggle:
1. Cell 2: Runtime controls (`SMOKE_TEST`, `DATASET_ROOT`, paper-faithful lock)
2. Cell 4: Imports
3. Cell 6: Config/device/profile
4. Cell 9: Dataset discovery
5. Cell 11: Train/val split
6. Cell 13: Dataset + dataloaders
7. Cell 15: Feature modules
8. Cell 17: Model definition
9. Cell 19: Loss/optimizer/epoch loop helpers
10. Cell 21: Training loop
11. Cell 23: Final evaluation
12. Cell 24: Extended dashboard (optional but recommended)
13. Cell 27: Export `.pth` artifact

Notes:
- Keep `FORCE_PAPER_FAITHFUL=True` for strict paper behavior.
- If your dataset is not auto-detected, set `DATASET_ROOT` in Cell 2.
- For quick validation, set `SMOKE_TEST=True`; for full run, set `SMOKE_TEST=False` and rerun from Cell 2 onward.

## Section 3: Dataset Discovery
### Block 3.1 - Auto-discover fake/real image paths and build dataframe

In [ ]:
# Kaggle-friendly dataset root discovery (balanced for smoke mode reliability)
fake_keywords = ['fake', 'deepfake', 'forged', 'manipulated', 'faceswap', 'synthetic', 'generated']
real_keywords = ['real', 'authentic', 'original', 'pristine', 'genuine']
img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def is_image(p: Path):
    return p.suffix.lower() in img_exts

def classify_dir_name(name: str):
    low = name.lower()
    if any(k in low for k in fake_keywords):
        return 0
    if any(k in low for k in real_keywords):
        return 1
    return None

def normalize_split_name(name: str):
    low = name.lower()
    if low in {'train', 'training'}:
        return 'train'
    if low in {'validation', 'valid', 'val'}:
        return 'val'
    if low in {'test', 'testing'}:
        return 'test'
    return None

def has_split_structure(root: Path):
    try:
        names = {p.name.lower() for p in root.iterdir() if p.is_dir()}
    except Exception:
        return False
    has_train = any(x in names for x in ['train', 'training'])
    has_val = any(x in names for x in ['validation', 'val', 'valid'])
    return has_train and has_val

def has_class_structure(root: Path):
    try:
        names = [p.name for p in root.iterdir() if p.is_dir()]
    except Exception:
        return False
    found = {classify_dir_name(n) for n in names}
    return 0 in found and 1 in found

def discover_dataset_roots(base_root: Path, max_depth=4):
    roots = []
    if not base_root.exists():
        return roots

    base_parts = len(base_root.parts)
    for d, dirs, _ in os.walk(base_root):
        cur = Path(d)
        depth = len(cur.parts) - base_parts
        if depth > max_depth:
            dirs[:] = []
            continue

        if has_split_structure(cur) or has_class_structure(cur):
            roots.append(cur)
            dirs[:] = []

    return roots

def find_class_dirs(root: Path, max_class_dirs_per_label=None):
    class_dirs = {0: [], 1: []}
    for d, _, _ in os.walk(root):
        label = classify_dir_name(Path(d).name)
        if label is None:
            continue

        if max_class_dirs_per_label is None or len(class_dirs[label]) < max_class_dirs_per_label:
            class_dirs[label].append(Path(d))

        if max_class_dirs_per_label is not None and (
            len(class_dirs[0]) >= max_class_dirs_per_label
            and len(class_dirs[1]) >= max_class_dirs_per_label
        ):
            break
    return class_dirs

def collect_from_dirs(class_dirs, max_scan_files=None, max_per_class=None):
    rows = []
    scanned = 0
    per_class = {0: 0, 1: 0}

    for label in [0, 1]:
        for dpath in class_dirs.get(label, []):
            if max_per_class is not None and per_class[label] >= max_per_class:
                break

            try:
                files = os.listdir(dpath)
            except Exception:
                continue

            for f in files:
                if max_scan_files is not None and scanned >= max_scan_files:
                    return rows
                if max_per_class is not None and per_class[label] >= max_per_class:
                    break

                p = dpath / f
                scanned += 1
                if p.is_file() and is_image(p):
                    rows.append((str(p), label))
                    per_class[label] += 1

    return rows

def clean_rows_to_df(rows):
    if not rows:
        return pd.DataFrame(columns=['path', 'label'])
    frame = pd.DataFrame(rows, columns=['path', 'label'])
    conflicts = frame.groupby('path')['label'].nunique()
    bad_paths = conflicts[conflicts > 1].index.tolist()
    if bad_paths:
        warnings.warn(f'Found {len(bad_paths)} conflicting path label(s); excluding them.')
        frame = frame[~frame['path'].isin(bad_paths)]
    return frame.drop_duplicates(subset=['path', 'label'])

def generate_smoke_data_if_needed(samples_per_class=40):
    smoke_root = Path('/kaggle/working/smoke_data' if is_kaggle else './smoke_data').resolve()
    for cls_name in ['fake', 'real']:
        d = smoke_root / cls_name
        d.mkdir(parents=True, exist_ok=True)
        for i in range(samples_per_class):
            arr = np.random.randint(0, 255, size=(cfg.img_size, cfg.img_size, 3), dtype=np.uint8)
            Image.fromarray(arr).save(d / f'{cls_name}_{i:03d}.png')
    print('Generated synthetic smoke dataset at:', smoke_root)
    class_dirs = find_class_dirs(smoke_root, max_class_dirs_per_label=10)
    return collect_from_dirs(class_dirs, max_scan_files=5000, max_per_class=MAX_SMOKE_PER_CLASS)

def build_candidate_roots():
    roots = []
    manual = Path(cfg.manual_data_root) if cfg.manual_data_root else None

    if manual is not None and manual.exists():
        roots.append(manual)
    elif manual is not None and not manual.exists():
        warnings.warn(f'DATASET_ROOT not found: {manual}. Falling back to auto-discovery under /kaggle/input.')

    if not roots and is_kaggle:
        auto_roots = discover_dataset_roots(Path('/kaggle/input'), max_depth=4)
        roots.extend(auto_roots)

    if not roots:
        roots.extend([p for p in [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()] if p.exists()])

    seen = set()
    roots = [p for p in roots if not (str(p) in seen or seen.add(str(p)))]
    return roots

def rebalance_smoke_dataframe(df_in: pd.DataFrame):
    if df_in.empty:
        return df_in

    df_bal = df_in.copy()
    counts = df_bal['label'].value_counts().to_dict()
    min_per_class = int(cfg.smoke_min_per_class)
    max_per_class = int(max(2, cfg.smoke_max_train // 2))

    need_more = {}
    for label in [0, 1]:
        current = int(counts.get(label, 0))
        if current < min_per_class:
            need_more[label] = min_per_class - current

    if need_more:
        synth_rows = generate_smoke_data_if_needed(samples_per_class=max(min_per_class, 40))
        synth_df = pd.DataFrame(synth_rows, columns=['path', 'label'])
        for label, needed in need_more.items():
            add_rows = synth_df[synth_df['label'] == label].head(needed)
            if not add_rows.empty:
                df_bal = pd.concat([df_bal, add_rows], ignore_index=True)

    capped = []
    for label in [0, 1]:
        part = df_bal[df_bal['label'] == label]
        if part.empty:
            continue
        take_n = min(len(part), max_per_class)
        capped.append(part.sample(n=take_n, random_state=cfg.seed))

    if len(capped) == 0:
        return df_bal

    return pd.concat(capped, ignore_index=True).drop_duplicates(subset=['path', 'label'])

MAX_SMOKE_PER_CLASS = 256
MAX_SCAN_FILES = 2000 if cfg.smoke_test else None
MAX_CLASS_DIRS_PER_LABEL = 20 if cfg.smoke_test else None

candidate_roots = build_candidate_roots()
all_rows = []
discovered_split_rows = {'train': [], 'val': [], 'test': []}

if cfg.smoke_test and not cfg.manual_data_root and not is_kaggle:
    all_rows = generate_smoke_data_if_needed()
else:
    for root in candidate_roots:
        if not root.exists():
            continue

        print('Scanning root:', root)

        # Prefer preserving existing train/val/test directory structure if available.
        if cfg.preserve_discovered_splits and has_split_structure(root):
            for child in root.iterdir():
                if not child.is_dir():
                    continue
                split_name = normalize_split_name(child.name)
                if split_name is None:
                    continue
                class_dirs = find_class_dirs(child, max_class_dirs_per_label=MAX_CLASS_DIRS_PER_LABEL)
                split_rows = collect_from_dirs(
                    class_dirs,
                    max_scan_files=MAX_SCAN_FILES,
                    max_per_class=(MAX_SMOKE_PER_CLASS if cfg.smoke_test else None),
                )
                discovered_split_rows[split_name].extend(split_rows)
            continue

        class_dirs = find_class_dirs(root, max_class_dirs_per_label=MAX_CLASS_DIRS_PER_LABEL)
        root_rows = collect_from_dirs(
            class_dirs,
            max_scan_files=MAX_SCAN_FILES,
            max_per_class=(MAX_SMOKE_PER_CLASS if cfg.smoke_test else None),
        )
        all_rows.extend(root_rows)

        if cfg.smoke_test:
            label_counts = pd.Series([r[1] for r in all_rows]).value_counts().to_dict() if all_rows else {}
            if label_counts.get(0, 0) >= MAX_SMOKE_PER_CLASS and label_counts.get(1, 0) >= MAX_SMOKE_PER_CLASS:
                break

# Build optional preserved split dataframes.
discovered_split_dfs = {}
for split_name in ['train', 'val', 'test']:
    split_df = clean_rows_to_df(discovered_split_rows.get(split_name, []))
    if not split_df.empty:
        discovered_split_dfs[split_name] = split_df

if len(all_rows) == 0 and cfg.smoke_test and not discovered_split_dfs:
    all_rows.extend(generate_smoke_data_if_needed())

if len(all_rows) == 0 and not discovered_split_dfs:
    checked = [f'{p} (exists={p.exists()})' for p in candidate_roots]
    raise RuntimeError(
        'No labeled fake/real image directories found. '
        'Set DATASET_ROOT to your dataset folder. '
        f'Checked roots: {checked}'
    )

df = clean_rows_to_df(all_rows)

if cfg.smoke_test and cfg.smoke_force_balanced and not df.empty:
    df = rebalance_smoke_dataframe(df)

if cfg.smoke_test and not discovered_split_dfs and len(df['label'].unique()) < 2:
    raise RuntimeError('Smoke test requires both classes; adjust DATASET_ROOT or smoke balancing settings.')

if discovered_split_dfs:
    print('Discovered preserved splits:', {k: len(v) for k, v in discovered_split_dfs.items()})
    for split_name, split_df in discovered_split_dfs.items():
        print(f"{split_name} label distribution:", split_df['label'].value_counts().to_dict())
else:
    print('Total discovered images:', len(df))
    print(df['label'].value_counts().to_dict())

df.head()

## Section 4: Data Splitting
### Block 4.1 - Create stratified train/validation partitions

In [ ]:
if cfg.preserve_discovered_splits and 'discovered_split_dfs' in globals() and {'train', 'val'}.issubset(set(discovered_split_dfs.keys())):
    train_df = discovered_split_dfs['train'].copy().reset_index(drop=True)
    val_df = discovered_split_dfs['val'].copy().reset_index(drop=True)
    test_df = discovered_split_dfs.get('test', pd.DataFrame(columns=['path', 'label'])).copy().reset_index(drop=True)
    print('Using preserved dataset splits from directory structure.')
else:
    train_df, val_df = train_test_split(
        df,
        test_size=cfg.val_size,
        random_state=cfg.seed,
        stratify=df['label']
    )
    test_df = pd.DataFrame(columns=['path', 'label'])

if cfg.smoke_test:
    train_take = max(1, cfg.smoke_max_train // 2)
    val_take = max(1, cfg.smoke_max_val // 2)

    train_parts = []
    val_parts = []
    for label in sorted(train_df['label'].unique().tolist()):
        tr_part = train_df[train_df['label'] == label]
        va_part = val_df[val_df['label'] == label]
        if not tr_part.empty:
            train_parts.append(tr_part.sample(n=min(len(tr_part), train_take), random_state=cfg.seed))
        if not va_part.empty:
            val_parts.append(va_part.sample(n=min(len(va_part), val_take), random_state=cfg.seed))

    train_df = pd.concat(train_parts, ignore_index=True) if train_parts else train_df
    val_df = pd.concat(val_parts, ignore_index=True) if val_parts else val_df

if len(train_df['label'].unique()) < 2 or len(val_df['label'].unique()) < 2:
    raise RuntimeError('Train/val split lost class diversity. Increase sample size or adjust dataset discovery.')

print('Train:', len(train_df), 'Val:', len(val_df), 'Test:', len(test_df))
print('Train label distribution:', train_df['label'].value_counts().to_dict())
print('Val label distribution:', val_df['label'].value_counts().to_dict())
if len(test_df) > 0:
    print('Test label distribution:', test_df['label'].value_counts().to_dict())

## Section 5: Input Pipeline
### Block 5.1 - Define transforms, dataset class, and dataloaders

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    # Paper-faithful augmentation: horizontal flip + zoom_range=0.2 only.
    transforms.RandomAffine(degrees=0, translate=None, scale=(0.8, 1.2), shear=None),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class DeepfakeImageDataset(Dataset):
    def __init__(self, frame_df: pd.DataFrame, transform=None, img_size=256):
        self.df = frame_df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row.path).convert('RGB')
        except (FileNotFoundError, UnidentifiedImageError, OSError) as e:
            warnings.warn(f'Failed to load image {row.path}: {e}')
            img = Image.new('RGB', (self.img_size, self.img_size), color=(127, 127, 127))

        if self.transform is not None:
            img = self.transform(img)

        label = torch.tensor(float(row.label), dtype=torch.float32)
        return img, label

train_ds = DeepfakeImageDataset(train_df, train_tfms, img_size=cfg.img_size)
val_ds = DeepfakeImageDataset(val_df, val_tfms, img_size=cfg.img_size)
test_ds = DeepfakeImageDataset(test_df, val_tfms, img_size=cfg.img_size) if 'test_df' in globals() and len(test_df) > 0 else None

pin_mem = bool(cfg.pin_memory and device.type == 'cuda')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=pin_mem)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=pin_mem)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=pin_mem) if test_ds is not None else None

print('Batches -> train:', len(train_loader), 'val:', len(val_loader), '| pin_memory:', pin_mem)
if test_loader is not None:
    print('Batches -> test:', len(test_loader))

## Section 6: Feature Extraction Modules
### Block 6.1 - Build Meso4 backbone, FPN fusion, and spatial attention

In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class FeaturePyramid(nn.Module):
    """PyTorch port of professor's FeaturePyramid block."""

    def __init__(self, in_ch=64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)

        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)

        self.conv_align1 = nn.Conv2d(64, 64, kernel_size=1, padding=0)
        self.conv_align2 = nn.Conv2d(128, 64, kernel_size=1, padding=0)
        self.conv_align3 = nn.Conv2d(256, 64, kernel_size=1, padding=0)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        c1 = self.relu(self.bn1(self.conv1(x)))
        c2 = self.relu(self.bn2(self.conv2(c1)))
        c3 = self.relu(self.bn3(self.conv3(c2)))

        c2_up = F.interpolate(c2, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        c3_up = F.interpolate(c3, size=c1.shape[-2:], mode='bilinear', align_corners=False)

        out = self.conv_align1(c1) + self.conv_align2(c2_up) + self.conv_align3(c3_up)
        return out


class SpatialAttention(nn.Module):
    """PyTorch port of professor's SpatialAttention block."""

    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn


class Meso4ProfessorBackbone(nn.Module):
    """Professor-faithful Meso4 stack: conv/bn/relu + stride-2 conv blocks, then FPN + spatial attention."""

    def __init__(self):
        super().__init__()

        self.b1a = ConvBNReLU(3, 8, k=3, stride=1, padding=1)
        self.b1b = ConvBNReLU(8, 8, k=3, stride=2, padding=1)

        self.b2a = ConvBNReLU(8, 16, k=3, stride=1, padding=1)
        self.b2b = ConvBNReLU(16, 16, k=3, stride=2, padding=1)

        self.b3a = ConvBNReLU(16, 32, k=3, stride=1, padding=1)
        self.b3b = ConvBNReLU(32, 32, k=3, stride=2, padding=1)

        self.b4a = ConvBNReLU(32, 64, k=3, stride=1, padding=1)
        self.b4b = ConvBNReLU(64, 64, k=3, stride=2, padding=1)

        self.fpn = FeaturePyramid(in_ch=64)
        self.sattn = SpatialAttention(kernel_size=7)

    def forward(self, x):
        x = self.b1a(x)
        x = self.b1b(x)

        x = self.b2a(x)
        x = self.b2b(x)

        x = self.b3a(x)
        x = self.b3b(x)

        x = self.b4a(x)
        x = self.b4b(x)

        x = self.fpn(x)
        x = self.sattn(x)
        return x

## Section 7: Hybrid Model Assembly
### Block 7.1 - Define capsule routing and FFAMPDV network

In [ ]:
class CapsuleLayer(nn.Module):
    """Dynamic routing capsule layer aligned to professor code semantics."""

    def __init__(self, input_num_capsules, input_dim_capsules, num_capsules=10, dim_capsules=16, routings=3):
        super().__init__()
        self.input_num_capsules = input_num_capsules
        self.input_dim_capsules = input_dim_capsules
        self.num_capsules = num_capsules
        self.dim_capsules = dim_capsules
        self.routings = routings

        self.W = nn.Parameter(
            torch.empty(
                input_num_capsules,
                num_capsules,
                input_dim_capsules,
                dim_capsules,
            )
        )
        nn.init.xavier_uniform_(self.W)

    @staticmethod
    def squash(s, eps=1e-8):
        s_norm = torch.sum(s * s, dim=-1, keepdim=True)
        return (s_norm / (1.0 + s_norm)) * (s / torch.sqrt(s_norm + eps))

    def forward(self, inputs):
        # inputs: [B, input_num_capsules, input_dim_capsules]
        if inputs.size(1) != self.input_num_capsules:
            raise ValueError(
                f'Capsule input token count mismatch: expected {self.input_num_capsules}, got {inputs.size(1)}. '
                f'Ensure img_size is fixed and matches model build time.'
            )

        # Equivalent to tf.einsum('bijc,ijcd->bijd', ...)
        inputs_hat = torch.einsum('bid,ijdf->bijf', inputs, self.W)

        b = torch.zeros(
            inputs_hat.size(0),
            self.input_num_capsules,
            self.num_capsules,
            device=inputs_hat.device,
            dtype=inputs_hat.dtype,
        )

        for i in range(self.routings):
            c = torch.softmax(b, dim=2)
            s = torch.sum(c.unsqueeze(-1) * inputs_hat, dim=1)
            v = self.squash(s)
            if i < self.routings - 1:
                b = b + torch.sum(inputs_hat * v.unsqueeze(1), dim=-1)

        return v


class SegformerFeatureExtractor(nn.Module):
    def __init__(self, model_name='mit_b0', pretrained=True, trainable=False, out_dim=64):
        super().__init__()
        self.using_timm = False
        self.fallback_dim = 32

        if TIMM_AVAILABLE:
            candidate_specs = []
            for name, family in [
                (model_name, 'segformer'),
                ('mit_b0', 'segformer'),
                ('mobilevitv2_100', 'alternate'),
                ('convnext_tiny', 'alternate'),
                ('efficientnet_b0', 'alternate'),
            ]:
                if name not in [n for n, _ in candidate_specs]:
                    candidate_specs.append((name, family))

            last_err = None
            for cand, family in candidate_specs:
                for use_pretrained in ([True, False] if pretrained else [False]):
                    try:
                        self.backbone = timm.create_model(
                            cand,
                            pretrained=use_pretrained,
                            num_classes=0,
                        )
                        self.using_timm = True
                        feat_dim = self.backbone.num_features
                        if family == 'segformer':
                            print(f'Using SegFormer/MiT backbone: {cand} (pretrained={use_pretrained})')
                        else:
                            print(f'Using alternate timm backbone: {cand} (pretrained={use_pretrained})')
                        break
                    except Exception as e:
                        last_err = e
                if self.using_timm:
                    break

            if not self.using_timm:
                warnings.warn(f'SegFormer init failed ({last_err}); using lightweight fallback branch.')
                self.backbone = nn.Sequential(
                    nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
                    nn.ReLU(inplace=True),
                    nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
                    nn.ReLU(inplace=True),
                    nn.AdaptiveAvgPool2d(1),
                )
                feat_dim = self.fallback_dim
        else:
            warnings.warn('timm not available; using lightweight fallback branch instead of SegFormer.')
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d(1),
            )
            feat_dim = self.fallback_dim

        if self.using_timm and not trainable:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.proj = nn.Linear(feat_dim, out_dim)

    def forward(self, x):
        if self.using_timm:
            f = self.backbone(x)
        else:
            f = self.backbone(x).flatten(1)
        return self.proj(f)


class FFAMPDVNet(nn.Module):
    """Professor-aligned FFAMPDV pipeline in PyTorch.

    Meso4 -> FeaturePyramid -> SpatialAttention -> Reshape(tokens,64) -> Capsule -> Flatten -> Dense(1)
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        self.backbone = Meso4ProfessorBackbone()

        token_hw = cfg.img_size // 16
        self.input_num_capsules = token_hw * token_hw

        self.caps = CapsuleLayer(
            input_num_capsules=self.input_num_capsules,
            input_dim_capsules=64,
            num_capsules=10,
            dim_capsules=16,
            routings=3,
        )

        self.segformer_branch = None
        segformer_dim = 0
        if cfg.use_segformer:
            self.segformer_branch = SegformerFeatureExtractor(
                model_name=cfg.segformer_variant,
                pretrained=cfg.segformer_pretrained,
                trainable=cfg.segformer_trainable,
                out_dim=64,
            )
            segformer_dim = 64

        self.classifier = nn.Sequential(
            nn.Linear(10 * 16 + segformer_dim, 1),
        )

    def forward(self, x):
        fm = self.backbone(x)  # [B,64,H/16,W/16]

        tokens = fm.flatten(2).transpose(1, 2)  # [B,N,64], equivalent to Reshape((-1,64))
        cap_feat = self.caps(tokens).flatten(1)  # [B,160]

        feats = [cap_feat]
        if self.segformer_branch is not None:
            feats.append(self.segformer_branch(x))

        feat = torch.cat(feats, dim=1)
        logit = self.classifier(feat).squeeze(1)
        return logit


model = FFAMPDVNet(cfg).to(device)
print('Model parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Section 8: Optimization and Epoch Logic
### Block 8.1 - Configure loss, optimizer, AMP scaler, and run_epoch

In [ ]:
if cfg.use_mse_loss:
    criterion = nn.MSELoss()
    print('Using MSELoss (paper mode).')
else:
    # Handle class imbalance for smoke/full runs when using BCE.
    class_counts = train_df['label'].value_counts().to_dict()
    neg_count = float(class_counts.get(0, 0))
    pos_count = float(class_counts.get(1, 0))
    if pos_count > 0 and neg_count > 0:
        pos_weight_value = max(1.0, neg_count / pos_count)
    else:
        pos_weight_value = 1.0
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    print(f'Using BCEWithLogitsLoss with pos_weight={pos_weight_value:.4f}')

try:
    # Force text-mode tqdm in notebooks to avoid widget-model errors.
    from tqdm.std import tqdm
except Exception:
    tqdm = None


def build_optimizer(model, cfg):
    name = cfg.optimizer_name.lower()
    params = [p for p in model.parameters() if p.requires_grad]

    if name == 'adam':
        return torch.optim.Adam(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    if name == 'sgd':
        return torch.optim.SGD(params, lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay, nesterov=True)
    return torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)


def build_scheduler(optimizer, cfg):
    name = cfg.scheduler_name.lower()
    if name == 'cosine':
        t_max = max(1, cfg.epochs)
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=cfg.min_lr)
    if name == 'plateau':
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1, min_lr=cfg.min_lr)
    return None


def build_amp_scaler(device):
    use_amp = device.type == 'cuda'
    # Torch 2.4+ prefers torch.amp.GradScaler('cuda', ...).
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=use_amp)
    return torch.cuda.amp.GradScaler(enabled=use_amp)


def amp_autocast(device):
    use_amp = device.type == 'cuda'
    # Torch 2.4+ prefers torch.amp.autocast('cuda', ...).
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast('cuda', enabled=use_amp)
    return torch.cuda.amp.autocast(enabled=use_amp)


optimizer = build_optimizer(model, cfg)
scheduler = build_scheduler(optimizer, cfg)
scaler = build_amp_scaler(device)


def _safe_classification_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        'acc': float('nan'),
        'precision': float('nan'),
        'recall': float('nan'),
        'f1': float('nan'),
        'roc_auc': float('nan'),
        'pr_auc': float('nan'),
    }

    if y_true.size == 0:
        return metrics, y_pred

    metrics['acc'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)

    if len(np.unique(y_true)) > 1 and len(np.unique(y_prob)) > 1:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
        metrics['pr_auc'] = average_precision_score(y_true, y_prob)

    return metrics, y_pred


def _is_invalid_loss(loss):
    return bool(torch.isnan(loss).item() or torch.isinf(loss).item())


def run_epoch(loader, train_mode=True, epoch_idx=1, total_epochs=None, batch_checkpoint_hook=None):
    model.train(train_mode)
    valid_losses = []
    y_true = []
    y_prob = []
    invalid_batches = 0

    phase = 'train' if train_mode else 'val'
    total_epochs = total_epochs or cfg.epochs
    iterator = loader
    if cfg.show_batch_progress and tqdm is not None:
        iterator = tqdm(
            loader,
            total=len(loader),
            desc=f'{phase} epoch {epoch_idx:02d}/{total_epochs:02d}',
            leave=False,
            dynamic_ncols=True,
            mininterval=0.7,
            bar_format='{l_bar}{bar:24}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]',
        )

    for batch_idx, (images, labels) in enumerate(iterator, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with amp_autocast(device):
                logits = model(images)
                if cfg.use_mse_loss:
                    probs_for_loss = torch.sigmoid(logits)
                    loss = criterion(probs_for_loss, labels)
                else:
                    loss = criterion(logits, labels)

            if _is_invalid_loss(loss):
                invalid_batches += 1
                warnings.warn(
                    f'{phase} batch {batch_idx}: NaN/Inf loss detected. Batch skipped from metrics and optimization.'
                )
                continue

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()

                if cfg.grad_clip_norm is not None and cfg.grad_clip_norm > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)

                scaler.step(optimizer)
                scaler.update()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        y_prob.extend(probs.tolist())
        y_true.extend(labels.detach().cpu().numpy().tolist())

        loss_value = loss.item()
        valid_losses.append(loss_value)
        if cfg.show_batch_progress and tqdm is not None and hasattr(iterator, 'set_postfix') and (batch_idx % 25 == 0):
            iterator.set_postfix_str(f'loss={loss_value:.4f} bad={invalid_batches}')

        if train_mode and batch_checkpoint_hook is not None:
            try:
                batch_checkpoint_hook(batch_idx)
            except Exception as e:
                warnings.warn(f'Batch checkpoint hook failed at batch {batch_idx}: {e}')

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)

    if len(valid_losses) == 0:
        raise RuntimeError('No valid batches were processed. Check dataset quality and model stability.')

    if invalid_batches > 0:
        warnings.warn(f'{phase} epoch {epoch_idx:02d}: skipped {invalid_batches} invalid batch(es).')

    metrics_core, y_pred = _safe_classification_metrics(y_true, y_prob, threshold=cfg.threshold)
    metrics = {
        'loss': float(np.mean(valid_losses)),
        'invalid_batches': int(invalid_batches),
        **metrics_core,
    }
    return metrics, y_true, y_prob, y_pred


## Section 9: Training
### Block 9.1 - Execute epoch loop and keep best checkpoint

In [ ]:
import time

history = []
best_val_auc = -1.0
best_val_score = -1.0
best_state = None
no_improve_epochs = 0

checkpoint_dir = Path('/kaggle/working/checkpoints' if is_kaggle else './checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_ckpt_path = checkpoint_dir / 'best_checkpoint.pth'
last_ckpt_path = checkpoint_dir / 'last_checkpoint.pth'


def _fmt_metric(v, digits=4):
    try:
        v = float(v)
    except Exception:
        return 'nan'
    if math.isnan(v) or math.isinf(v):
        return 'nan'
    return f'{v:.{digits}f}'


def select_primary_val_metric(val_metrics):
    roc_auc = val_metrics.get('roc_auc', float('nan'))
    pr_auc = val_metrics.get('pr_auc', float('nan'))
    f1 = val_metrics.get('f1', float('nan'))

    if not math.isnan(roc_auc):
        return roc_auc, 'roc_auc'
    if not math.isnan(pr_auc):
        return pr_auc, 'pr_auc'
    if not math.isnan(f1):
        return f1, 'f1'
    return float('-inf'), 'none'


def _safe_scheduler_state_dict(scheduler_obj):
    if scheduler_obj is None:
        return None
    try:
        return scheduler_obj.state_dict()
    except Exception:
        return None


def save_last_checkpoint(epoch, interrupted=False, phase='epoch_end', batch_idx=0):
    payload = {
        'epoch': int(epoch),
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': _safe_scheduler_state_dict(scheduler),
        'scaler_state_dict': scaler.state_dict(),
        'history': history,
        'best_val_auc': float(best_val_auc),
        'best_val_score': float(best_val_score),
        'no_improve_epochs': int(no_improve_epochs),
        'config': cfg.__dict__,
        'interrupted': bool(interrupted),
        'phase': str(phase),
        'batch_idx': int(batch_idx),
        'saved_at': time.time(),
    }
    torch.save(payload, last_ckpt_path)


start_epoch = 1
if cfg.resume_training and last_ckpt_path.exists():
    try:
        resume_ckpt = torch.load(last_ckpt_path, map_location='cpu')
        model.load_state_dict(resume_ckpt['state_dict'])
        if 'optimizer_state_dict' in resume_ckpt:
            optimizer.load_state_dict(resume_ckpt['optimizer_state_dict'])
        if scheduler is not None and resume_ckpt.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(resume_ckpt['scheduler_state_dict'])
        if 'scaler_state_dict' in resume_ckpt:
            scaler.load_state_dict(resume_ckpt['scaler_state_dict'])

        history = resume_ckpt.get('history', [])
        best_val_auc = float(resume_ckpt.get('best_val_auc', -1.0))
        best_val_score = float(resume_ckpt.get('best_val_score', -1.0))
        no_improve_epochs = int(resume_ckpt.get('no_improve_epochs', 0))

        resume_phase = str(resume_ckpt.get('phase', 'epoch_end'))
        resume_epoch = int(resume_ckpt.get('epoch', 0))
        if resume_phase == 'train_batch':
            # Snapshot happened mid-epoch, so restart that same epoch.
            start_epoch = max(1, resume_epoch)
            print(f'Resumed from mid-epoch snapshot at epoch {resume_epoch}, restarting that epoch.')
        else:
            start_epoch = resume_epoch + 1
            print(f'Resumed training from epoch {start_epoch}.')

        if best_ckpt_path.exists():
            try:
                best_ckpt = torch.load(best_ckpt_path, map_location='cpu')
                if 'state_dict' in best_ckpt:
                    best_state = {k: v.cpu() for k, v in best_ckpt['state_dict'].items()}
            except Exception:
                best_state = None
    except Exception as e:
        warnings.warn(f'Could not resume from last checkpoint: {e}. Starting from epoch 1.')
        start_epoch = 1

train_start_time = time.time()

try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        if cfg.max_train_hours > 0:
            elapsed_hours = (time.time() - train_start_time) / 3600.0
            if elapsed_hours >= cfg.max_train_hours:
                print(f'Max train time reached ({cfg.max_train_hours}h). Stopping at epoch {epoch - 1}.')
                break

        epoch_start = time.time()

        def _batch_snapshot_hook(batch_idx):
            if cfg.checkpoint_every_n_batches <= 0:
                return
            if batch_idx % cfg.checkpoint_every_n_batches == 0:
                save_last_checkpoint(epoch, interrupted=False, phase='train_batch', batch_idx=batch_idx)

        tr_metrics, _, _, _ = run_epoch(
            train_loader,
            train_mode=True,
            epoch_idx=epoch,
            total_epochs=cfg.epochs,
            batch_checkpoint_hook=_batch_snapshot_hook,
        )

        should_run_val = (epoch % cfg.val_every_n_epochs == 0) or (epoch == cfg.epochs)
        if should_run_val:
            va_metrics, y_true_val, y_prob_val, y_pred_val = run_epoch(
                val_loader,
                train_mode=False,
                epoch_idx=epoch,
                total_epochs=cfg.epochs,
            )
        else:
            va_metrics = {
                'loss': float('nan'),
                'invalid_batches': 0,
                'acc': float('nan'),
                'precision': float('nan'),
                'recall': float('nan'),
                'f1': float('nan'),
                'roc_auc': float('nan'),
                'pr_auc': float('nan'),
            }

        primary_score, primary_name = select_primary_val_metric(va_metrics)

        if scheduler is not None:
            if cfg.scheduler_name.lower() == 'plateau':
                scheduler_key = primary_score
                if scheduler_key == float('-inf'):
                    val_loss = va_metrics.get('loss', float('nan'))
                    scheduler_key = (-val_loss) if not math.isnan(val_loss) else None
                if scheduler_key is not None:
                    scheduler.step(scheduler_key)
            else:
                scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        row = {'epoch': epoch, 'lr': current_lr, 'primary_metric': primary_name}
        row.update({f'train_{k}': v for k, v in tr_metrics.items()})
        row.update({f'val_{k}': v for k, v in va_metrics.items()})
        history.append(row)

        improved = should_run_val and (primary_score > best_val_score)
        if improved:
            best_val_score = primary_score
            current_auc = va_metrics['roc_auc']
            if not math.isnan(current_auc):
                best_val_auc = current_auc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            no_improve_epochs = 0

            best_ckpt = {
                'epoch': epoch,
                'best_val_auc': float(best_val_auc),
                'best_val_score': float(best_val_score),
                'best_val_primary_metric': primary_name,
                'state_dict': best_state,
                'config': cfg.__dict__,
            }
            torch.save(best_ckpt, best_ckpt_path)
            status = 'BEST'
        else:
            if should_run_val:
                no_improve_epochs += 1
                status = f'WAIT({no_improve_epochs})'
            else:
                status = 'TRAIN_ONLY'

        epoch_sec = time.time() - epoch_start
        print(
            f"Ep {epoch:02d}/{cfg.epochs:02d} | {status} | {primary_name}={_fmt_metric(primary_score)} | "
            f"lr={current_lr:.2e} | "
            f"train: loss={_fmt_metric(tr_metrics['loss'])} auc={_fmt_metric(tr_metrics['roc_auc'])} f1={_fmt_metric(tr_metrics['f1'])} | "
            f"val: loss={_fmt_metric(va_metrics['loss'])} auc={_fmt_metric(va_metrics['roc_auc'])} pr={_fmt_metric(va_metrics['pr_auc'])} f1={_fmt_metric(va_metrics['f1'])} | "
            f"bad={tr_metrics['invalid_batches']}/{va_metrics['invalid_batches']} | {epoch_sec:.1f}s"
        )

        save_last_checkpoint(epoch, interrupted=False, phase='epoch_end', batch_idx=0)

        # Release cached objects aggressively on long runs to reduce memory pressure.
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        if should_run_val and epoch >= cfg.min_epochs_before_stop and no_improve_epochs >= cfg.early_stopping_patience:
            print(f'Early stopping triggered at epoch {epoch:02d}/{cfg.epochs:02d}.')
            break

except KeyboardInterrupt:
    # Save latest state so long full-dataset runs can be resumed after notebook/session interrupts.
    interrupted_epoch = history[-1]['epoch'] if len(history) > 0 else max(1, start_epoch - 1)
    save_last_checkpoint(interrupted_epoch, interrupted=True, phase='interrupt', batch_idx=0)
    print('Training interrupted by KeyboardInterrupt. Saved resumable state to:', last_ckpt_path)

history_df = pd.DataFrame(history)
history_df.tail()


## Section 10: Validation and Visualization
### Block 10.1 - Compute final metrics, confusion matrix, ROC and PR curves

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

eval_loader = test_loader if 'test_loader' in globals() and test_loader is not None else val_loader
eval_split_name = 'test' if eval_loader is test_loader else 'validation'

final_metrics, y_true_val, y_prob_val, y_pred_val = run_epoch(
    eval_loader,
    train_mode=False,
    epoch_idx=min(len(history), cfg.epochs),
    total_epochs=cfg.epochs,
    )
print(f'Final {eval_split_name} metrics:', final_metrics)

cm = confusion_matrix(y_true_val, y_pred_val)
print('Confusion matrix:\n', cm)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

if len(np.unique(y_true_val)) > 1 and len(np.unique(y_prob_val)) > 1:
    RocCurveDisplay.from_predictions(y_true_val, y_prob_val, ax=axs[0])
    axs[0].set_title('ROC Curve')

    PrecisionRecallDisplay.from_predictions(y_true_val, y_prob_val, ax=axs[1])
    axs[1].set_title('Precision-Recall Curve')
else:
    axs[0].text(0.5, 0.5, 'ROC not available\n(single class or constant probs)', ha='center', va='center')
    axs[1].text(0.5, 0.5, 'PR not available\n(single class or constant probs)', ha='center', va='center')
    axs[0].set_title('ROC Curve')
    axs[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

plot_dir = Path('/kaggle/working' if is_kaggle else './outputs')
plot_dir.mkdir(parents=True, exist_ok=True)
plot_path = plot_dir / f'{eval_split_name}_curves.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
print('Saved plot:', plot_path)

In [ ]:
# Section 10.2 - Extended performance dashboard and sample predictions
# Label convention in this notebook: 0 = Fake, 1 = Real
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
)


def _safe_div(a, b):
    return float(a) / float(b) if b else float('nan')


# Reuse existing validation outputs when available; otherwise run one val pass.
if 'y_true_val' not in globals() or 'y_prob_val' not in globals():
    eval_metrics, y_true_val, y_prob_val, y_pred_val = run_epoch(
        val_loader,
        train_mode=False,
        epoch_idx=min(len(history), cfg.epochs) if 'history' in globals() else 1,
        total_epochs=cfg.epochs,
    )

# Ensure numpy arrays for metric computation.
y_true_eval = np.asarray(y_true_val).astype(int)
y_prob_eval = np.asarray(y_prob_val).astype(float)
y_pred_eval = (y_prob_eval >= cfg.threshold).astype(int)

cm_eval = confusion_matrix(y_true_eval, y_pred_eval, labels=[0, 1])
if cm_eval.shape == (2, 2):
    tn, fp, fn, tp = cm_eval.ravel()
else:
    tn = fp = fn = tp = 0

ext_metrics = {
    'threshold': float(cfg.threshold),
    'accuracy': accuracy_score(y_true_eval, y_pred_eval),
    'balanced_accuracy': balanced_accuracy_score(y_true_eval, y_pred_eval),
    'precision': precision_score(y_true_eval, y_pred_eval, zero_division=0),
    'recall_sensitivity': recall_score(y_true_eval, y_pred_eval, zero_division=0),
    'specificity_tnr': _safe_div(tn, tn + fp),
    'f1': f1_score(y_true_eval, y_pred_eval, zero_division=0),
    'mcc': matthews_corrcoef(y_true_eval, y_pred_eval),
    'roc_auc': roc_auc_score(y_true_eval, y_prob_eval) if len(np.unique(y_true_eval)) > 1 and len(np.unique(y_prob_eval)) > 1 else float('nan'),
    'pr_auc': average_precision_score(y_true_eval, y_prob_eval) if len(np.unique(y_true_eval)) > 1 and len(np.unique(y_prob_eval)) > 1 else float('nan'),
    'fpr': _safe_div(fp, fp + tn),
    'fnr': _safe_div(fn, fn + tp),
    'npv': _safe_div(tn, tn + fn),
    'tp': int(tp),
    'tn': int(tn),
    'fp': int(fp),
    'fn': int(fn),
}

metrics_df = pd.DataFrame({'metric': list(ext_metrics.keys()), 'value': list(ext_metrics.values())})
print('Extended validation metrics (0=Fake, 1=Real):')
display(metrics_df)

# Curves and confusion matrix visualization.
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Confusion matrix plot
ax_cm = axs[0, 0]
im = ax_cm.imshow(cm_eval, cmap='Blues')
ax_cm.set_title('Confusion Matrix')
ax_cm.set_xlabel('Predicted')
ax_cm.set_ylabel('True')
ax_cm.set_xticks([0, 1])
ax_cm.set_xticklabels(['Fake (0)', 'Real (1)'])
ax_cm.set_yticks([0, 1])
ax_cm.set_yticklabels(['Fake (0)', 'Real (1)'])
for i in range(cm_eval.shape[0]):
    for j in range(cm_eval.shape[1]):
        ax_cm.text(j, i, f'{cm_eval[i, j]}', ha='center', va='center', color='black')
fig.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.04)

# ROC curve
ax_roc = axs[0, 1]
if len(np.unique(y_true_eval)) > 1 and len(np.unique(y_prob_eval)) > 1:
    fpr_arr, tpr_arr, _ = roc_curve(y_true_eval, y_prob_eval)
    ax_roc.plot(fpr_arr, tpr_arr, label=f"ROC AUC={ext_metrics['roc_auc']:.4f}")
    ax_roc.plot([0, 1], [0, 1], '--', color='gray')
    ax_roc.legend(loc='lower right')
else:
    ax_roc.text(0.5, 0.5, 'ROC unavailable', ha='center', va='center')
ax_roc.set_title('ROC Curve')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')

# Precision-Recall curve
ax_pr = axs[1, 0]
if len(np.unique(y_true_eval)) > 1 and len(np.unique(y_prob_eval)) > 1:
    prec_arr, rec_arr, _ = precision_recall_curve(y_true_eval, y_prob_eval)
    ax_pr.plot(rec_arr, prec_arr, label=f"PR AUC={ext_metrics['pr_auc']:.4f}")
    ax_pr.legend(loc='lower left')
else:
    ax_pr.text(0.5, 0.5, 'PR unavailable', ha='center', va='center')
ax_pr.set_title('Precision-Recall Curve')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')

# Threshold sweep chart for quick decision-threshold insight
ax_th = axs[1, 1]
thr = np.linspace(0.05, 0.95, 19)
f1_vals, p_vals, r_vals = [], [], []
for t in thr:
    yp = (y_prob_eval >= t).astype(int)
    f1_vals.append(f1_score(y_true_eval, yp, zero_division=0))
    p_vals.append(precision_score(y_true_eval, yp, zero_division=0))
    r_vals.append(recall_score(y_true_eval, yp, zero_division=0))
ax_th.plot(thr, f1_vals, label='F1')
ax_th.plot(thr, p_vals, label='Precision')
ax_th.plot(thr, r_vals, label='Recall')
ax_th.axvline(cfg.threshold, linestyle='--', color='gray', label=f'Current={cfg.threshold:.2f}')
ax_th.set_title('Threshold Sweep')
ax_th.set_xlabel('Threshold')
ax_th.set_ylabel('Score')
ax_th.legend()

plt.tight_layout()
plt.show()

# Show sample images with true vs predicted labels and confidence.
label_name = {0: 'Fake', 1: 'Real'}
sample_n = min(8, len(val_df))
if sample_n > 0:
    sample_df = val_df.sample(n=sample_n, random_state=cfg.seed).reset_index(drop=True)
    model.eval()

    rows = int(np.ceil(sample_n / 4))
    cols = min(4, sample_n)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    elif cols == 1:
        axes = np.array([[ax] for ax in axes])

    sample_records = []
    for idx, row in sample_df.iterrows():
        r = idx // cols
        c = idx % cols
        ax = axes[r, c]

        true_label = int(row['label'])
        try:
            pil_img = Image.open(row['path']).convert('RGB')
            inp = val_tfms(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                logit = model(inp).squeeze(0)
                prob_real = float(torch.sigmoid(logit).item())
            pred_label = int(prob_real >= cfg.threshold)

            ax.imshow(pil_img)
            title = f"T:{label_name[true_label]} | P:{label_name[pred_label]} ({prob_real:.2f})"
            ax.set_title(title, color=('green' if pred_label == true_label else 'red'), fontsize=10)
            ax.axis('off')

            sample_records.append({
                'path': row['path'],
                'true_label': label_name[true_label],
                'pred_label': label_name[pred_label],
                'prob_real': prob_real,
                'correct': bool(pred_label == true_label),
            })
        except Exception as e:
            ax.text(0.5, 0.5, f'Load/Infer failed\n{e}', ha='center', va='center')
            ax.axis('off')

    # Hide any unused subplot slots.
    for k in range(sample_n, rows * cols):
        rr = k // cols
        cc = k % cols
        axes[rr, cc].axis('off')

    plt.suptitle('Sample Predictions (True vs Predicted)', y=1.02)
    plt.tight_layout()
    plt.show()

    if sample_records:
        sample_pred_df = pd.DataFrame(sample_records)
        print('Sample prediction table:')
        display(sample_pred_df)
else:
    print('No validation samples available for image-level preview.')

## Section 11: Model Export
### Block 11.1 - Save trained artifact to .pth for Kaggle output

### Export Reminder

Cell 25 can auto-recover `final_metrics` by running a quick validation pass if evaluation was skipped.
For best results, still run Cell 22 (and optionally Cell 23) before export.

In [ ]:
profile_slug = cfg.training_profile.lower().replace(' ', '_')
arch_slug = 'segformer' if cfg.use_segformer else 'baseline'
model_title = 'FFA-MPDV-inspired-with-SegFormer' if cfg.use_segformer else 'FFA-MPDV-paper-baseline'

# Recover metrics automatically if the evaluation cell was skipped.
if 'final_metrics' not in globals() or final_metrics is None:
    print('`final_metrics` not found. Running quick validation pass for export metadata...')
    export_loader = None
    if 'test_loader' in globals() and test_loader is not None:
        export_loader = test_loader
    elif 'val_loader' in globals() and val_loader is not None:
        export_loader = val_loader

    if export_loader is None:
        raise RuntimeError('No validation/test loader available to recover `final_metrics`. Run data/eval cells first.')
    final_metrics, _, _, _ = run_epoch(
        export_loader,
        train_mode=False,
        epoch_idx=min(len(history), cfg.epochs) if 'history' in globals() else cfg.epochs,
        total_epochs=cfg.epochs,
    )

# Prefer exporting the best checkpoint weights if available.
state_to_export = model.state_dict()
if 'best_state' in globals() and best_state is not None:
    state_to_export = best_state
    print('Exporting best checkpoint weights from memory.')

history_to_export = history if 'history' in globals() else []

artifact = {
    'model_name': model_title,
    'model_class': 'FFAMPDVNet',
    'state_dict': state_to_export,
    'config': cfg.__dict__,
    'history': history_to_export,
    'final_metrics': final_metrics,
    'notes': 'Saved as .pth only (no .pkl). Requires FFAMPDVNet definition for loading.',
}


def _candidate_export_dirs():
    dirs = []

    if is_kaggle:
        dirs.extend([
            Path('/kaggle/working/outputs'),
            Path('/kaggle/working'),
            Path('/kaggle/output'),
        ])

    # Local/portable fallbacks.
    dirs.extend([
        Path.cwd() / 'outputs',
        Path.cwd(),
        Path('./outputs').resolve(),
    ])

    # Keep order, remove duplicates.
    seen = set()
    uniq = []
    for d in dirs:
        key = str(d.resolve()) if d is not None else ''
        if key not in seen:
            seen.add(key)
            uniq.append(d)
    return uniq


def _save_with_fallback(payload, filename):
    errors = []
    for d in _candidate_export_dirs():
        try:
            d.mkdir(parents=True, exist_ok=True)
            out = d / filename
            torch.save(payload, out)
            if out.exists() and out.stat().st_size > 0:
                print('Saved model to:', out.resolve())
                print('File size (MB):', round(out.stat().st_size / (1024 * 1024), 3))
                return out
        except Exception as e:
            errors.append(f'{d}: {e}')

    msg = 'Model export failed for all candidate directories.\n' + '\n'.join(errors)
    raise RuntimeError(msg)


filename = f'ffa_mpdv_{arch_slug}_{profile_slug}_deepfake_detector.pth'
out_path = _save_with_fallback(artifact, filename)

# Optional extra copy for Kaggle output pane if available.
if is_kaggle:
    kaggle_output_dir = Path('/kaggle/output')
    if kaggle_output_dir.exists() and out_path.parent.resolve() != kaggle_output_dir.resolve():
        try:
            kaggle_copy = kaggle_output_dir / out_path.name
            torch.save(artifact, kaggle_copy)
            print('Kaggle output copy:', kaggle_copy.resolve())
        except Exception as e:
            print('Kaggle output copy failed:', e)

print('Final export path:', out_path.resolve())